In [1]:
import pandas as pd
from pathlib import Path
import numpy as np
from itertools import chain
import string

from chem_informed_mtgp.robot_recipe_utils import batch_by_temp_then_time

In [4]:
from notebooks.configs.paths import DATA_ROOT as project_path

new_batch_df = pd.read_csv(
    project_path / "BO_results_outlier_removed" / "20260803_shap_minmax.csv",
    comment="#",
)
print(
    new_batch_df.drop(columns=["a_site", "a_site_smiles"]).to_markdown(
        tablefmt="github"
    )
)

|    |   annealing_temp |   annealing_time |   dispense_volume |   spin_speed |   dispense_speed |   concentration |   SHAP_Combined_Env |
|----|------------------|------------------|-------------------|--------------|------------------|-----------------|---------------------|
|  0 |              125 |              630 |               100 |         3500 |              160 |            0.25 |            0.959434 |
|  1 |               80 |              630 |               100 |         3500 |              160 |            0.5  |            0.805961 |
|  2 |              130 |              630 |               100 |         3500 |              160 |            1    |           -6.35429  |
|  3 |              135 |              630 |               100 |         3500 |              160 |            0.25 |           13.4502   |
|  4 |              130 |              630 |               100 |         3500 |              160 |            1    |           -9.27694  |
|  5 |               80 |  

In [ ]:
# duplicate everthing 4 time to get 8 samples each
new_batch_df = pd.concat([new_batch_df] * 4, ignore_index=True)
print(
    new_batch_df.drop(columns=["a_site", "a_site_smiles"]).to_markdown(
        tablefmt="github"
    )
)

In [7]:
new_batch_df_1_index = (new_batch_df["a_site"] == "ClMBAI") | (
    (new_batch_df["a_site"] == "MBAI") & (new_batch_df["concentration"] == 0.25)
)
new_batch_df = new_batch_df[~new_batch_df_1_index].reset_index(drop=True)

In [5]:
new_batch_df.groupby(["a_site"]).agg("count")

,annealing_temp,annealing_time,dispense_volume,spin_speed,dispense_speed,concentration,a_site_smiles,SHAP_Combined_Env
a_site,,,,,,,,
ClMBAI,2,2,2,2,2,2,2,2
MBAI,3,3,3,3,3,3,3,3
MeOMBAI,3,3,3,3,3,3,3,3


In [6]:
# BATCH_SIZE = 4
BATCH_SIZE = 8

try:
    # Apply the optimization
    new_batch_optimized = batch_by_temp_then_time(
        new_batch_df,
        batch_size=BATCH_SIZE,
        max_temp_range=[(40, 70), (70, 150)],
    )

    # Verify the results
    print("Batch ranges:")
    total_time_cost = 0

    for i in range(0, len(new_batch_optimized), BATCH_SIZE):
        batch = new_batch_optimized.iloc[i : i + BATCH_SIZE]
        temps = batch["annealing_temp"].values
        times = batch["annealing_time"].values
        time_cost = np.max([times.max(), times.min() + 3 * 120]) + 120 * 5 + 250 * 8
        print(
            f"Batch {i // 4 + 1}: temp_span: {temps.min()}-{temps.max()} °C, "
            f"time_range: {times.max()}-{times.min()} s, time_cost={time_cost / 60:.1f} min"
        )
        total_time_cost += time_cost

    print(f"Total time cost for all batches: {total_time_cost / 60:.1f} minutes")

except ValueError as e:
    print(e)
    # Don't overwrite new_x_df_optimized with garbage if failed, to preserve state

new_batch_optimized

Batch ranges:
Batch 1: temp_span: 80.0-135.0 °C, time_range: 630-630 s, time_cost=59.8 min
Total time cost for all batches: 59.8 minutes


,annealing_temp,annealing_time,dispense_volume,spin_speed,dispense_speed,concentration,a_site,a_site_smiles,SHAP_Combined_Env
0,135.0,630,100.0,3500.0,160.0,1.00,MBAI,C[C@@H](C1=CC=C(C=C1))N,-8.535904
1,130.0,630,100.0,3500.0,160.0,0.25,MBAI,C[C@@H](C1=CC=C(C=C1))N,4.437959
2,80.0,630,100.0,3500.0,160.0,0.50,MBAI,C[C@@H](C1=CC=C(C=C1))N,6.448525
3,130.0,630,100.0,3500.0,160.0,1.00,ClMBAI,C[C@@H](C1=CC=C(C=C1)Cl)N,-9.276937
4,135.0,630,100.0,3500.0,160.0,0.25,ClMBAI,C[C@@H](C1=CC=C(C=C1)Cl)N,13.450202
5,130.0,630,100.0,3500.0,160.0,1.00,MeOMBAI,C[C@@H](C1=CC=C(C=C1)OC)N,-6.354293
6,80.0,630,100.0,3500.0,160.0,0.50,MeOMBAI,C[C@@H](C1=CC=C(C=C1)OC)N,0.805961
7,125.0,630,100.0,3500.0,160.0,0.25,MeOMBAI,C[C@@H](C1=CC=C(C=C1)OC)N,0.959434


In [13]:
# carrier_chunk_size = 4
# n_repeats_per_recipe = 2

carrier_chunk_size = 8
n_repeats_per_recipe = 1


carrier_chunks = [
    new_batch_optimized.iloc[i : i + carrier_chunk_size].copy()  # ensure it's a copy
    for i in range(0, len(new_batch_optimized), carrier_chunk_size)
]

recipe_chunks = list(
    chain(*[[chunk] * n_repeats_per_recipe for chunk in carrier_chunks])
)
new_recipe_df = pd.concat(recipe_chunks, ignore_index=True)
new_recipe_df["wash"] = "CHCl3"

recipe_dir = project_path / "LPS-32"

if not recipe_dir.exists():
    recipe_dir.mkdir(parents=True)

new_batch_optimized.to_csv(
    Path(recipe_dir, "proposed_total_correlation_xs.csv"), index=False
)
new_recipe_df.to_json(
    Path(recipe_dir, "total_correlation_recipes.json"), orient="records", indent=2
)

In [14]:
molecular_weights = {
    "MeOMBAI": 279.12,
    "MBAI": 249.08,
    "ClMBAI": 283.53,
    "PbI2": 461.01,
}

# with old vial holder 1 from sciprios
# vial_loc = ["1-21", "1-20", "1-19", "1-3", "1-2", "1-1", "2-19", "2-20", "2-21", "2-1", "2-2", "2-3"]

# new vial holder 1: holds 1-1 to 1-23
vial_loc = [f"1-{i}" for i in range(1, 24)]


VIAL_CAPACITY = 1225
MIM_ROUNDUP = 50
ROUND_UP_TO_NEAREST = 25
MAX_ROUNDUP = MIM_ROUNDUP + ROUND_UP_TO_NEAREST

PbI2_conc = 0.5
PbI2_vol_per_sample = 40

# Get the unique [a_site, concentration] combinations for the proposed batch, get the combined volume for each
precursor_by_vials = (
    new_recipe_df.groupby(["a_site", "concentration"]).agg(
        dispense_volume=("dispense_volume", "sum"),
        sample_count=("dispense_volume", "count"),
    )
).reset_index()

precursor_by_vials["sample_count"] = (
    precursor_by_vials["sample_count"] / n_repeats_per_recipe
).astype("int")  # because each recipe is repeated twice

base_concentration = (
    precursor_by_vials.groupby("a_site")["concentration"].agg("max").reset_index()
)

# precursor_by_a_site = precursor_by_vials.groupby("a_site")['dispense_volume'].agg('sum').reset_index()
for a_site, group in precursor_by_vials.groupby("a_site"):
    base_conc = base_concentration.loc[
        base_concentration["a_site"] == a_site, "concentration"
    ].values[0]
    for idx, row in group.iterrows():
        conc = row["concentration"]
        vol = row["dispense_volume"]

        # add 30 to vol and snap to nearest 25 uL for pipetting convenience
        vol_roundup = (
            np.ceil((vol + MIM_ROUNDUP) / ROUND_UP_TO_NEAREST) * ROUND_UP_TO_NEAREST
        )

        if vol_roundup > VIAL_CAPACITY:
            vials_split = int(np.ceil(vol_roundup / (VIAL_CAPACITY - MIM_ROUNDUP)))
            vol_roundup = (
                np.ceil((vol / vials_split + MIM_ROUNDUP) / ROUND_UP_TO_NEAREST)
                * ROUND_UP_TO_NEAREST
            ) * vials_split  # recalculate vol_roundup based on split volume
            print(
                f"Warning: total volume {vol} for {conc} M {a_site} exceeds max vial volume, splitted into {vials_split} vials with {(vol_roundup) / vials_split} uL each"
            )

        base_solution_vol = (
            np.round(vol_roundup * (conc / base_conc), 1)
            if not np.isclose(conc, base_conc)
            else vol_roundup
        )

        precursor_by_vials.loc[
            idx, ["vol_roundup", "base_solution_vol", "solvent_vol"]
        ] = [vol_roundup, base_solution_vol, vol_roundup - base_solution_vol]

precursor_by_vials["label"] = list(string.ascii_lowercase)[: len(precursor_by_vials)]


vial_loc_iter = iter(vial_loc)
try:
    vial_loc_column = []
    for idx, row in precursor_by_vials.iterrows():
        vial_locs = [
            next(vial_loc_iter)
            for _ in range(
                np.ceil(
                    row["vol_roundup"] / (VIAL_CAPACITY + ROUND_UP_TO_NEAREST)
                ).astype(int)
            )
        ]
        vial_loc_column.append(vial_locs)

except StopIteration:
    total_vials_needed = sum(
        (precursor_by_vials["vol_roundup"] // 1200).astype(int) + 1
    )
    raise ValueError(
        f"Not enough vial locations for precursors: {len(precursor_by_vials)} precursors requires {total_vials_needed} vial locations but only {len(vial_loc)} given."
    )

precursor_by_vials["vial_locs"] = vial_loc_column

precursor_by_vials

,a_site,concentration,dispense_volume,sample_count,vol_roundup,base_solution_vol,solvent_vol,label,vial_locs
0,ClMBAI,0.25,100.0,1,150.0,37.5,112.5,a,[1-1]
1,ClMBAI,1.00,100.0,1,150.0,150.0,0.0,b,[1-2]
2,MBAI,0.25,100.0,1,150.0,37.5,112.5,c,[1-3]
3,MBAI,0.50,100.0,1,150.0,75.0,75.0,d,[1-4]
4,MBAI,1.00,100.0,1,150.0,150.0,0.0,e,[1-5]
5,MeOMBAI,0.25,100.0,1,150.0,37.5,112.5,f,[1-6]
6,MeOMBAI,0.50,100.0,1,150.0,75.0,75.0,g,[1-7]
7,MeOMBAI,1.00,100.0,1,150.0,150.0,0.0,h,[1-8]


In [15]:
precursor_by_a_site = precursor_by_vials.groupby("a_site")[
    ["concentration", "base_solution_vol"]
].agg(
    concentration=("concentration", "max"),
    base_solution_vol=("base_solution_vol", "sum"),
)

precursor_by_a_site.loc["PbI2", ["concentration", "base_solution_vol"]] = [
    0.5,
    np.ceil((PbI2_vol_per_sample * len(new_recipe_df) + 70) / 25)
    * 25,  # add 70 uL for pipetting convenience before rounding,
]

precursor_by_a_site.loc["PbI2", "base_solution_vol"] += (
    100 if precursor_by_a_site.loc["PbI2", "base_solution_vol"] > 1150 else 0
)

precursor_by_a_site["base_solution_vol_roundup"] = precursor_by_a_site[
    "base_solution_vol"
].apply(lambda x: np.ceil((x + 50) / 25) * 25)

precursor_by_a_site = precursor_by_a_site.assign(
    chemical_mass=lambda df: np.round(
        df["base_solution_vol_roundup"]
        * df["concentration"]
        * df.index.map(molecular_weights)
        * 1e-3,
        1,
    )
)

precursor_by_a_site

,concentration,base_solution_vol,base_solution_vol_roundup,chemical_mass
a_site,,,,
ClMBAI,1.0,187.5,250.0,70.9
MBAI,1.0,262.5,325.0,81.0
MeOMBAI,1.0,262.5,325.0,90.7
PbI2,0.5,400.0,450.0,103.7


In [16]:
import pandas

DMSO_MW = 78.13
DMSO_DENSITY = 1.1

DMSO_1_equiv_vol_from_PbI2_weight_factor = (
    1 / molecular_weights["PbI2"] * DMSO_MW / DMSO_DENSITY
)

DMSO_vol = (
    precursor_by_a_site.loc["PbI2", "chemical_mass"]
    * DMSO_1_equiv_vol_from_PbI2_weight_factor
)

IPA_for_base_solution_vol = (
    precursor_by_a_site["base_solution_vol_roundup"].sum()
    - precursor_by_a_site.loc["PbI2", "base_solution_vol_roundup"]
)

solvents = pandas.DataFrame(
    {
        "solvent": ["IPA", "DMF", "DMSO"],
        "volume": [
            precursor_by_vials["solvent_vol"].sum() + IPA_for_base_solution_vol,
            precursor_by_a_site.loc["PbI2", "base_solution_vol_roundup"] - DMSO_vol,
            DMSO_vol,
        ],
    }
)

solvents["volume"] = np.round(solvents["volume"], 1)

solvents

,solvent,volume
0,IPA,1387.5
1,DMF,434.0
2,DMSO,16.0


In [17]:
out_path = Path(recipe_dir) / "solutions.html"

# -----------------------------------------------------------------
#  HTML template - everything is built as one long string for
#  clarity.  Only the CSS part is new compared to your original code.
# -----------------------------------------------------------------
html = f"""<!DOCTYPE html>
<html>
<head>
    <meta charset="utf-8">
    <!-- Keep the classic Jupyter-notebook look -->
    <link rel="stylesheet"
          href="https://cdn.jupyter.org/notebook/5.1.0/style/style.min.css">

    <style>
        /* -------------------------------------------------
           Full-page wrapper - takes the whole viewport height
           ------------------------------------------------- */
        .page-wrapper {{
            min-height: 100vh;                 /* fill the window */
            display: flex;
            flex-direction: column;            /* stack children vertically */
        }}

        /* -------------------------------------------------
           The block that holds the two lower tables.
           It will shrink-wrap to the width the tables need,
           and the auto-margins centre it vertically.
           ------------------------------------------------- */
        .side-by-side {{
            display: inline-flex;               /* side-by-side, but DON'T stretch */
            gap: 20px;                          /* breathing room between tables */
            margin-top: 20px;                   /* push it down */
            margin-bottom: auto;                /* push it up   → centre vertically */
            align-self: center;                 /* centre horizontally (optional) */
        }}

        /* Children keep their intrinsic width - no flex grow/shrink */
        .side-by-side > div {{
            flex: none;                         /* stay at natural width */
            overflow-x: auto;                   /* scroll if a table is too wide */
        }}

        /* Make each inner table fill the width of its wrapper */
        .side-by-side table {{
            width: auto;                        /* let the content decide */
            border-collapse: collapse;
        }}

        /* Responsive fallback - on very narrow screens stack them */
        @media (max-width: 600px) {{
            .side-by-side {{
                display: block;                 /* stack vertically */
                text-align: left;
            }}
            .side-by-side > div {{
                margin-bottom: 15px;
            }}
        }}
    </style>
</head>
<body class="rendered_html">

<div class="page-wrapper">
    <h2>Recipe of {recipe_dir.stem}</h2>

    <!-- Full-width first table -->
    {precursor_by_vials.to_html()}

    <!-- Lower two tables - side-by-side, shrink-wrapped, vertically centred -->
    <div class="side-by-side">
        <div>
            {precursor_by_a_site.reset_index().to_html(index=False)}
        </div>
        <div>
            {solvents.to_html(index=False)}
        </div>
    </div>
</div>

</body>
</html>
"""

# -----------------------------------------------------------------
#  Write the file
# -----------------------------------------------------------------
with open(out_path, "w", encoding="utf-8") as f:
    f.write(html)

precursor_by_vials[
    ["a_site", "concentration", "vial_locs", "label", "sample_count"]
].to_csv(recipe_dir / "precursor_by_vials.csv", index=False)

In [ ]:
import json
from ast import literal_eval
import yaml
from datetime import datetime
import copy
import pandas as pd
from pathlib import Path

from notebooks.configs.paths import batch_dir


# old vial holder 1 from sciprios
# solution_num_PbI2_vial = [22]

# new vial holder 1: 1-24 reserved for PbI2
solution_num_PbI2_vial = [24]

recipe_dir = batch_dir(32)


def get_balanced_vial(num_recipes: int, vial_list: list, current_recipe_index: int):
    """
    Distributes recipes across vials as evenly as possible (AAAABBBCCC).
    Handles remainders by adding 1 to the first few groups.
    """
    num_vials = len(vial_list)
    if num_vials == 0:
        raise ValueError("vial_list cannot be empty.")

    base_count = num_recipes // num_vials
    remainder = num_recipes % num_vials

    # The first 'remainder' vials get (base_count + 1) items
    cutoff = remainder * (base_count + 1)

    if current_recipe_index < cutoff:
        vial_idx = current_recipe_index // (base_count + 1)
    else:
        vial_idx = remainder + (current_recipe_index - cutoff) // base_count

    return vial_list[vial_idx]


# --- Configuration ---
reverse_carrier_order = False  # Set to True to reverse the order of carriers
# Load the recipes from the JSON file
json_path = Path(recipe_dir, "total_correlation_recipes.json")
with open(json_path, "r") as f:
    recipes = json.load(f)

# --- Load Concentration to Vial Mapping ---
conc_to_solution_map = {}
conc_vial_path = recipe_dir / "precursor_by_vials.csv"
if not conc_vial_path.exists():
    if (recipe_dir / "conc-to-vial.txt").exists():
        raise NotImplementedError(
            "Please use the old version in BO.ipynb to generate the robot recipe with .txt mapping file."
        )
    else:
        raise FileNotFoundError(
            f"Concentration to vial mapping file not found: {conc_vial_path}"
        )

conc_df = pd.read_csv(conc_vial_path, header=0)


def eval_vial_loc(x):
    if "[" in x and "]" in x:
        res = literal_eval(x)
        if isinstance(res, list):
            return res
        else:
            raise ValueError(f"vial_loc value '{x}' is not a list")
    else:
        return [x]  # Wrap non-list values in a list for consistency


def split_to_vial_station_and_solution_idx(vials):
    return list(map(lambda x: tuple(x.split("-")), vials))


conc_df["vial_locs"] = (
    conc_df["vial_locs"]
    .apply(eval_vial_loc)
    .apply(split_to_vial_station_and_solution_idx)
)
conc_df["sample_count"] = conc_df["sample_count"] * 2
conc_to_solution_map = conc_df.set_index(["concentration", "a_site"])
conc_to_solution_map["allocated_count"] = (
    0  # Track how many times each concentration has been allocated to recipes
)

# Define output YAML file path
date_str = datetime.now().strftime("%Y%m%d")
yaml_filename = f"{date_str}-{recipe_dir.stem}.yaml"
yaml_path = Path(recipe_dir, yaml_filename)

# Mapping from dispense_speed (uL/s) to Liquid Class Index Dispensing
# Based on the provided example and parameter ranges.
dispense_speed_to_lci = {
    80: 60,
    100: 61,
    120: 62,
    140: 63,
    160: 64,
    180: 65,
    200: 66,
    220: 67,
    240: 68,
}

# --- Base Templates ---
# Template for the fixed PbI2 layer (step_1)
step_1_template = {
    "Process": "Spin Coating",
    "Substrate_1": {
        "Annealing Station Number": 2,
        "Annealing Temperature (°C)": 70,
        "Annealing Time (s)": 60,
        "Discard Pipettes": True,
        "Dispense-Spin/Blade Delay (s)": 3.0,
        "Liquid Class Index Aspiration": 82,
        "Liquid Class Index Dispensing": 82,
        "Mixing Cycles Aspiration (0-9)": 0,
        "Pipette Dispensing Height Above Substrate (mm)": 1.5,
        "Pipette Rack Number": 1,
        "Put on Hotplate": True,
        "Solution Number": 22,
        "Solution Volume (uL)": 40,
        "Solvent Station Number": 1,
        "Spin Acceleration (rpm/s)": 1000,
        "Spin Time (s)": 30,
        "Spin Velocity (rpm)": 2000,
        "Spincoater Number": 2,
        "Tip Type": "300 µL",
    },
}
# Populate step_1 for all 8 substrates
for i in range(2, 9):
    step_1_template[f"Substrate_{i}"] = step_1_template["Substrate_1"].copy()

# Template for the variable perovskite layer (step_2)
step_2_substrate_template = {
    "Annealing Station Number": 1,
    "Annealing Temperature (°C)": 60,  # Placeholder
    "Annealing Time (s)": 1500,  # Placeholder
    "Arm Speed While Dispending (0-100%)": 0,
    "Discard Pipettes": True,
    "Dispense range (mm)": 0,
    "Dispense-Spin/Blade Delay (s)": 3.0,
    "Initial Spin Velocity (Before Dispensing) (rpm)": 1000,  # Placeholder
    "Liquid Class Index Aspiration": 60,
    "Liquid Class Index Dispensing": 65,  # Placeholder
    "Mixing Cycles Aspiration (0-9)": 0,
    "Pipette Dispensing Height Above Substrate (mm)": 1.5,
    "Pipette Rack Number": 1,
    "Put on Hotplate": True,
    "Solution Number": 21,  # Placeholder, will be determined by concentration
    "Solution Volume (uL)": 125,  # Placeholder
    "Solvent Station Number": 1,
    "Spin Acceleration (rpm/s)": 1000,
    "Spin Time (s)": 30,
    "Spin Velocity (rpm)": 1000,  # Placeholder
    "Spincoater Number": 1,
    "Tip Type": "300 µL",
}

# --- YAML Generation ---
final_yaml_data = {}
num_carriers = (
    len(recipes) + 7
) // 8  # Calculate needed carriers (8 substrates per carrier)

sample_to_vial_allocation = []  # Track which sample gets allocated to which vial for debugging

for i in range(num_carriers):
    carrier_key = f"carrier_{i + 1}"
    # Use a deepcopy to prevent YAML anchors/aliases
    final_yaml_data[carrier_key] = {"step_1": copy.deepcopy(step_1_template)}

    step_2_data = {"Process": "Whirl Spin Coating"}
    start_idx = i * 8
    end_idx = start_idx + 8
    carrier_recipes = recipes[start_idx:end_idx]

    # Find the minimum annealing temperature for the current carrier
    # Ensure hotplate at least 50°C, Autobot cannot deal with SV < PV, safer to set a floor as TEC also generates heat
    min_annealing_temp = max(50, min(int(r["annealing_temp"]) for r in carrier_recipes))

    for j, recipe in enumerate(carrier_recipes):
        substrate_key = f"Substrate_{j + 1}"
        final_yaml_data[carrier_key]["step_1"][substrate_key]["Solution Number"] = (
            get_balanced_vial(len(recipes), solution_num_PbI2_vial, start_idx + j)
        )

        substrate_data = step_2_substrate_template.copy()
        # Update data from the JSON recipe
        substrate_data["Annealing Temperature (°C)"] = min_annealing_temp
        substrate_data["Annealing Time (s)"] = int(recipe["annealing_time"])
        substrate_data["Solution Volume (uL)"] = int(recipe["dispense_volume"])
        substrate_data["Initial Spin Velocity (Before Dispensing) (rpm)"] = int(
            recipe["spin_speed"]
        )
        substrate_data["Spin Velocity (rpm)"] = int(recipe["spin_speed"])

        # Map dispense speed to LCI
        dispense_speed = int(recipe["dispense_speed"])
        try:
            substrate_data["Liquid Class Index Dispensing"] = dispense_speed_to_lci[
                dispense_speed
            ]
        except KeyError as e:
            raise KeyError(
                f"Error: Dispense speed {dispense_speed} uL/s not found in mapping."
            ) from e

        # Map concentration to Solution Number
        concentration = float(recipe["concentration"])

        try:
            num_samples, vial_locs, allocated_count = conc_to_solution_map.loc[
                (concentration, recipe["a_site"]),
                ["sample_count", "vial_locs", "allocated_count"],
            ]
            station_number, solution_number = get_balanced_vial(
                num_samples, vial_locs, allocated_count
            )
            substrate_data["Solution Number"] = int(solution_number)
            substrate_data["Solvent Station Number"] = int(station_number)
            # Increment the allocated count for this concentration
            conc_to_solution_map.at[
                (concentration, recipe["a_site"]), "allocated_count"
            ] += 1
        except KeyError as e:
            a_site_text = f"for a_site '{a_site}'" if "a_site" in recipe else ""
            raise KeyError(
                f"Error: Concentration {concentration} M {a_site_text} not found in conc-to-vial.txt."
            ) from e

        step_2_data[substrate_key] = substrate_data
        sample_to_vial_allocation.append(
            {
                "sample": f"{i + 1}-{j + 1}",
                "precursor": f"{concentration} M {recipe['a_site']}",
                "allocated_vial": f"{station_number}-{solution_number}",
            }
        )

    final_yaml_data[carrier_key]["step_2"] = step_2_data

# --- Reverse carrier order if flag is set ---
if reverse_carrier_order:
    reversed_items = reversed(list(final_yaml_data.items()))
    final_yaml_data = {f"carrier_{i + 1}": v for i, (k, v) in enumerate(reversed_items)}

# --- Save to File ---
with open(yaml_path, "w", encoding="utf-8") as f:
    # Add allow_unicode=True to prevent escaping special characters
    yaml.dump(
        final_yaml_data, f, sort_keys=False, indent=2, width=1000, allow_unicode=True
    )

print(f"Successfully generated YAML file at: {yaml_path}")

pd.DataFrame(sample_to_vial_allocation).groupby("allocated_vial").agg(set)

Successfully generated YAML file at: G:\Shared drives\Sutter-Fella Lab\ECRP-Project\Yi-Ru\manuscripts\LPS-data-zenodo\LPS-32\20261104-LPS-32.yaml


,sample,precursor
allocated_vial,,
1-1,{1-5},{0.25 M ClMBAI}
1-2,{1-4},{1.0 M ClMBAI}
1-3,{1-2},{0.25 M MBAI}
1-4,{1-3},{0.5 M MBAI}
1-5,{1-1},{1.0 M MBAI}
1-6,{1-8},{0.25 M MeOMBAI}
1-7,{1-7},{0.5 M MeOMBAI}
1-8,{1-6},{1.0 M MeOMBAI}
